<a href="https://colab.research.google.com/github/KuberVNaik712/BIS-Lab/blob/main/week6(_Cuckoo_Search_Algorithm).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import numpy as np
import math

traffic_demand = np.array([900, 600, 1200, 400])
saturation_flow = np.array([1800, 1800, 1800, 1800])

n = 25
Pa = 0.25
MaxIter = 200
alpha = 0.5
beta = 1.5

CYCLE_TIME = 120
LOST_TIME = 16
MIN_GREEN = 10
MAX_GREEN = 60

AVAILABLE_GREEN = CYCLE_TIME - LOST_TIME


def fitness(x):
    total_delay = 0

    for i in range(4):
        g = x[i]
        demand = traffic_demand[i]
        capacity = saturation_flow[i] * g / CYCLE_TIME

        ratio = demand / capacity
        green_ratio = g / CYCLE_TIME

        denominator = max(
            1 - min(ratio, 0.99) * green_ratio,
            0.01
        )

        delay = (
            0.5 * CYCLE_TIME *
            (1 - green_ratio) ** 2 / denominator
        )

        if ratio > 0.95:
            delay += 1000 * (ratio - 0.95) ** 2

        total_delay += demand * delay

    return total_delay


def levy_flight():
    sigma = (
        math.gamma(1 + beta) *
        np.sin(np.pi * beta / 2) /
        (
            math.gamma((1 + beta) / 2) *
            beta *
            2 ** ((beta - 1) / 2)
        )
    ) ** (1 / beta)

    u = np.random.normal(0, sigma, 4)
    v = np.random.normal(0, 1, 4)

    return u / (np.abs(v) ** (1 / beta) + 1e-10)


def repair(x):
    x = np.maximum(x, MIN_GREEN)

    for _ in range(100):
        x = np.clip(x, MIN_GREEN, MAX_GREEN)

        diff = AVAILABLE_GREEN - np.sum(x)

        if abs(diff) < 1e-6:
            break

        if diff > 0:
            free = np.where(x < MAX_GREEN - 1e-6)[0]
        else:
            free = np.where(x > MIN_GREEN + 1e-6)[0]

        if len(free) == 0:
            break

        x[free] += diff / len(free)

    return x


nests = np.array([
    repair(np.random.uniform(MIN_GREEN, MAX_GREEN, 4))
    for _ in range(n)
])

fitness_values = np.array([
    fitness(x) for x in nests
])

best_index = np.argmin(fitness_values)

best_nest = nests[best_index].copy()
best_fitness = fitness_values[best_index]

t = 0

while t < MaxIter:

    for i in range(n):

        step = levy_flight()

        new_solution = nests[i] + alpha * step

        new_solution = repair(new_solution)

        new_fitness = fitness(new_solution)

        j = np.random.randint(n)

        if new_fitness < fitness_values[j]:

            nests[j] = new_solution
            fitness_values[j] = new_fitness

    abandon_count = int(Pa * n)

    worst_nests = np.argsort(fitness_values)[-abandon_count:]

    for j in worst_nests:

        new_nest = np.random.uniform(
            MIN_GREEN,
            MAX_GREEN,
            4
        )

        new_nest = repair(new_nest)

        nests[j] = new_nest
        fitness_values[j] = fitness(new_nest)

    best_index = np.argmin(fitness_values)

    if fitness_values[best_index] < best_fitness:

        best_fitness = fitness_values[best_index]
        best_nest = nests[best_index].copy()

    t += 1


print("\n========== OPTIMIZED TRAFFIC SIGNAL ==========")

directions = ["North", "South", "East", "West"]

for i in range(4):
    print(
        directions[i],
        "Green Time:",
        round(best_nest[i], 2),
        "seconds"
    )

print("\nTotal Cycle Time:", CYCLE_TIME, "seconds")
print("Lost Time:", LOST_TIME, "seconds")
print("Available Green Time:", round(sum(best_nest), 2), "seconds")
print("Best Fitness:", round(best_fitness, 2))



========== OPTIMIZED TRAFFIC SIGNAL ==========
North Green Time: 30.22 seconds
South Green Time: 20.15 seconds
East Green Time: 40.24 seconds
West Green Time: 13.39 seconds

Total Cycle Time: 120 seconds
Lost Time: 16 seconds
Available Green Time: 104.0 seconds
Best Fitness: 3473893.73
